# Do surface linguistic features predict hostility in Spanish comments?

Hate-speech classifiers usually run on embeddings, which work well and explain nothing.
This experiment asks a narrower, more interpretable question: **can a handful of cheap
surface features — computed with a single spaCy pass and no training — separate hostile
comments from ordinary ones?**

The corpus is a set of Spanish news-site comments, each annotated with an `INTENSIDAD`
(intensity) score. A comment is treated as hostile when its intensity is above zero.

Four features are proposed, each with a prior reason to expect a difference:

| Feature | Hypothesis |
|---|---|
| Verb ratio | Hostile messages lean on imperatives and direct address |
| Adjective ratio | Insults are largely adjectival |
| Uppercase-word ratio | Shouting is written in capitals |
| Emotional punctuation ratio | `!` and `?` carry the affect |

The point is not to build a classifier. It is to find out **which of these intuitions
survive contact with data** — measured as an effect size, not as a difference in means
that noise alone could produce.

All the machinery lives in `src/surface_features/` and is covered by the test suite;
this notebook is the narrative that drives it.

## Setup

The package is imported from `src/`; the Spanish model must be installed:

```bash
pip install -r requirements.txt
python -m spacy download es_core_news_md
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from surface_features import experiment
from surface_features.corpus import read_comments
from surface_features.spacy_backend import load_pipeline, parse
from surface_features.stats import format_table

## Data

The corpus is not in this repository (~391 MB); see the README for how to obtain and
clean it. It is streamed row by row rather than loaded into a dataframe: three of its
columns are used, and streaming keeps peak memory proportional to one row instead of to
the whole file.

`LIMIT` keeps the notebook re-runnable in a couple of minutes. Set it to `None` for the
full corpus.

In [ ]:
CSV = "comentarios_limpio_utf8.csv"
LIMIT = 50_000

comments = []
for comment in read_comments(CSV):
    comments.append(comment)
    if LIMIT is not None and len(comments) >= LIMIT:
        break

hostile = sum(c.hostile for c in comments)
print(f"comments read  {len(comments):,}")
print(f"hostile        {hostile:,} ({hostile / len(comments) * 100:.1f}%)")

## Feature extraction

One pass over the corpus, one parse per comment, every feature derived from it.

Every feature is a **ratio**, not a count. This matters more than it looks: hostile
comments could simply be longer, and a raw adjective count would then measure length
rather than hostility. Dividing by the word-token count removes that confound — and the
length columns are reported alongside, so the confound stays visible rather than being
assumed away.

Only the tagger is loaded. Disabling the parser and the rest is not a micro-optimisation:
on a corpus this size the parser alone roughly doubles wall-clock time for information no
feature reads.

In [ ]:
pipeline = load_pipeline()  # es_core_news_md, tagger only
documents = parse(pipeline, (c.text for c in comments))

data = experiment.build(documents, (c.hostile for c in comments))
print(f"usable comments {len(data):,}  (dropped {data.dropped:,} with no word tokens)")

Comments that are nothing but emoji or punctuation are **dropped**, not scored as zero.
A row of zeros would not mark them missing — it would assert that they contain no verbs
and no adjectives, and thousands of such rows drag every group mean toward zero. The
dropped count is printed rather than hidden.

Labels are consumed alongside documents, so a dropped comment takes its label with it.
Extracting features first and zipping labels afterwards shifts every label by the number
of dropped rows and produces plausible, entirely wrong results.

## Is any of this real?

A difference in means is not evidence on its own — with tens of thousands of comments,
almost any difference reaches statistical significance while remaining far too small to
act on. The useful question is **effect size**: how large is the gap relative to the
spread within each group?

Cohen's *d* answers that. The conventional reading is 0.2 small, 0.5 medium, 0.8 large;
this table adds a `negligible` band below 0.1, where the distributions overlap so heavily
that no classifier can exploit the difference. A column with no within-group spread is
reported as `undefined` rather than as zero, and sorted last: an undefined statistic must
never head the ranking.

In [ ]:
effects = experiment.effects(data)
print(format_table(effects))

## Reading the result

Whatever sits at the top is the only surface feature worth feeding to a classifier. The
`negligible` rows are intuitions that did not survive — the more interesting half of the
result, because each one is a plausible-sounding feature that would have added dimensions
and no signal.

Two caveats worth stating rather than burying:

- **The label is a threshold on a continuous score.** `INTENSIDAD > 0` collapses mild and
  extreme hostility into one class, compressing exactly the differences this experiment is
  trying to measure.
- **Ratios are unstable on short comments.** A three-word comment with one adjective
  scores 0.33, far outside anything a longer comment can reach. The length columns are in
  the table above so this is visible rather than assumed away.

## Do the surviving features actually classify?

The honest test of a feature set is whether a model built only on it beats the base rate.
On an imbalanced corpus, accuracy cannot answer that: a model predicting "not hostile"
everywhere already scores the base rate. So the random forest is compared against an
explicit majority-class baseline, and ROC-AUC is the number that matters — 0.5 means the
features carry nothing.

In [ ]:
result = experiment.classify(data, seed=0)
print(f"majority-class accuracy  {result.majority_accuracy:.3f}")
print(f"random forest accuracy   {result.model_accuracy:.3f}")
print(f"random forest ROC-AUC    {result.roc_auc:.3f}")
print(f"beats the baseline       {result.beats_baseline}")

In [ ]:
for name, value in experiment.importances(data, seed=0):
    print(f"{name:<24}{value:.4f}")

## Conclusion

The experiment set out to test four intuitions about hostile language using features cheap
enough to compute in a single pass. Where the effect sizes and the feature importances
agree on the ranking, that is the reassuring outcome — two different methods, one
distributional and one model-based, picking out the same features.

The broader point: surface features are interpretable and nearly free, but they are a
*floor*, not a solution. They capture how something is written, not what it says, and a
hostile comment written in calm lowercase prose defeats every feature here. That is the
gap embeddings exist to fill — and knowing the size of the gap is worth more than
assuming it.

`examples/synthetic_corpus.py` runs the same pipeline over two synthetic groups with a
known planted difference, with no corpus and no model required. If that script cannot
recover what was planted, nothing in this notebook can be trusted.